## Simulation


This notebook demonstrates the workflow for running a set of 2D Navier-Stokes (CFD) simulations, extracting results for machine learning and generating datasets for training surrogate models.   

The code below:
- sets up simulation parameters
- runs PyFR CFD cases across different configurations
- processes the data for downstream ML tasks

Next --> main-training.ipynb for building the surrogate model!

In [1]:
import numpy as np
import pandas as pd
import itertools

from pathlib import Path

from process_results import process_sim_results
from extract_training_data import extract_all_cases
from run_sim import PyfrSimulation
from animation import gen_mp4
from plot_static_results import plot_column

ROOT = Path().resolve().parent

SIM_NAME = "example-model"

### Build Cases

In [2]:
# Set parameter ranges
nu_rng = (0.01, 0.02)      # [nu, m/s^2] kintematic velocity, 0.01 for Re ~ 100 @ 1 Uin
uin_rng = (1.0, 2.0)       # [Uin, m/s] inlet velocity, free-stream velocity (nondim)
dt_rng = (0.05, 0.10)      # [dt, s] solver timestep (adjust if unstable)
tend_rng = (10.0, 20.0)    # [tend, s] total time (long enough for a few vortex shedding cycles)
dt_out_rng = (0.05, 0.10)  # [dt_out] save state interval
STEPS = 1                  # number of steps per parameter min/max

# Build cases
param_ranges = [
    np.linspace(*nu_rng, STEPS),
    np.linspace(*uin_rng, STEPS),
    np.linspace(*dt_rng, STEPS),
    np.linspace(*tend_rng, STEPS),
    np.linspace(*dt_out_rng, STEPS),
]

cases = [list(params) for params in itertools.product(*param_ranges)]
len(cases)

1

### Run CFD for Training Data

In [ ]:
%%time

# run simulation
MESH_FILE = "../assets/config/2d-cylinder.msh"
PYFRM_FILE = "../assets/config/2d-cylinder.pyfrm"
INI_FILE = "../assets/config/2d-cylinder.ini"

m = PyfrSimulation(SIM_NAME, mesh_file=MESH_FILE, pyfrm_file=PYFRM_FILE)
m.run_bulk(cases, backend="metal", show_progress=True)

# Note, it can take several minutes per case.

New ini file: /Volumes/connor/dev/projects/ns2d-surrogate/sims/example-model/config/case0.ini
Running 1 simulations...


  0%|          | 0/1 [00:00<?, ?it/s]

### Process Results

In [ ]:
# Convert VTKs to PVD files for easier post-processing
process_sim_results(SIM_NAME)

In [ ]:
# # Extract nodewise results for a single simulation case to CSV for training data
df = extract_all_cases(SIM_NAME)
display(df.head())

# del df  # Free memory
df = df.to_pandas()  # plots still on pandas

### Visualise Raw Results

In [ ]:
# Generate MP4 for simulation results (optional)
FPS = 20                     # Frames per second
CMAP = "turbo"               # Color map
OFF_SCREEN = True            # Render offscreen
WINDOW_SIZE = (1920, 1088)   # Resolution of rendered frames (width, height) 

for i in range(len(cases)):
    print(f"Generating animation for Case: 'case{i}'")
    gen_mp4(
        SIM_NAME, 
        f"case{i}", 
        remove_images=True,
        fps=FPS,
        cmap=CMAP,
        off_screen=OFF_SCREEN,
        window_size=WINDOW_SIZE
    )

# MP4s saved in 'sims/<sim_name>/animations/<sim_name>_<case_idx>.mp4'

In [ ]:
# Visualise static results
CASE = "case0" 
STEP = 11
METRIC = "p" # p, u, vn
    
if 'df' not in locals() and 'df' not in globals():
    results_path = ROOT / "sims" / SIM_NAME / "training_data" / f"{CASE}-results.csv"
    df = pd.read_csv(results_path)

plot_column(df, METRIC, case_name=CASE, step=STEP)